# AMAZON PRODUCT SCRAPER

# Scraping Amazon books, Prices, Links

In [ ]:
from selenium import webdriver
import pandas as pd
#imported the required libraries

driver = webdriver.Chrome('C:/Users/visha/OneDrive/Desktop/python/chromedriver_win32/chromedriver.exe') #connecting to the chromedriver app required for scraping
driver.get('https://www.amazon.in/s?i=stripbooks&bbn=976390031&rh=n%3A976389031%2Cn%3A1318158031%2Cp_n_feature_three_browse-bin%3A9141482031&dc&page=75&qid=1624637686&rnid=976390031&ref=sr_pg_74') #url for scraping 

listing = {} #empty dictionary for product names
price = {} #empty dictiuonary for prices 
link = {}
#using dictionary because some items do not have prices. we can eliminate these items by utilizing keys which are available for dictionaries and not lists.

#xpath for all the product names
listing_web_elm= driver.find_elements_by_xpath("//*[@class = 's-main-slot s-result-list s-search-results sg-row']//*[@class='a-link-normal a-text-normal']") 

#xpath for all the prices
price_web_elm = driver.find_elements_by_xpath("//*[@class = 's-main-slot s-result-list s-search-results sg-row']//*[@class='a-size-base a-link-normal a-text-normal']") 

for j in listing_web_elm: #for loop to get all the product names
    item_name=j.text
    item_href=j.get_attribute('href') #href for creating a link between product and price
    #if 'Noodles' in item_name and 'Sunfeast' in item_name:
    listing[item_href]=item_name
 
    
for k in price_web_elm: #for loop to get all the prices
    item_name=k.text
    item_href=k.get_attribute('href')
    item_name=item_name.split(' ')[0][1:] #selects only the final price and ignores the currency symbol, discount, etc.
    price[item_href]=item_name 
    
#the href link ensures that a product's name and price are taken only if both of them exist
    
pairs=[] #empty list to store everything scraped 


for key in price:
    if key in listing:
        pairs.append([listing[key],price[key],key]) #adding the scraped data to the list
    
    
    
amazon = pd.DataFrame(pairs, columns=["Item","Price","Link"]) #storing the data in a dataframe and also renaming the columns

amazon.to_csv(r'C:\Users\visha\OneDrive\Desktop\python\Amazon Action Adventure Books NotCleaned.csv', mode='a', index = False, header=None)

# Scraping Amazon Ratings and Reviews

In [ ]:
from selenium import webdriver
import pandas as pd
from tqdm import tqdm
#imported the required libraries



def revpage(href):
    driver = webdriver.Chrome('C:/Users/visha/OneDrive/Desktop/python/chromedriver_win32/chromedriver.exe')
    href = href.replace("/dp/","/product-reviews/")
    href = href.replace("%2Fdp%","%2Fproduct-reviews%")
    driver.get(href)
    comment = []
    comments = driver.find_elements_by_xpath("//*[@class = 'a-fixed-right-grid-col a-col-left']//*[@class='a-row a-spacing-small review-data']")

    for m in comments: 
        comment.append(m.text)
    driver.close()
    if comment == []:
        return ""
    return(comment)

def revrate(href):
    driver = webdriver.Chrome('C:/Users/visha/OneDrive/Desktop/python/chromedriver_win32/chromedriver.exe')
    href = href.replace("/dp/","/product-reviews/")
    href = href.replace("%2Fdp%","%2Fproduct-reviews%")
    driver.get(href)
    rating = []
    ratings = driver.find_elements_by_xpath("//*[@class = 'a-fixed-right-grid-col a-col-left']//*[@class='a-size-medium a-color-base']")

    for n in ratings: 
        rating.append(n.text)
    driver.close()
    if rating == []:
        return ""
    return(rating)
   
amproduct = pd.read_csv('Amazon Action Adventure Books NotCleaned.csv')

amratings = []
amcomments = []
for url in tqdm(amproduct['Link']):
        amcomments.append(revpage(url))
        amratings.append(revrate(url))



# Exporting the data collected

In [ ]:
import pandas as pd
df = pd.read_csv(r'C:\Users\visha\OneDrive\Desktop\python\Amazon Action Adventure Books NotCleaned.csv')
len(df)

In [ ]:
df['Rating'] = amratings
df['Comments'] = amcomments
df

In [ ]:
df.to_excel('Amazon Action Adventure Books Cleaned.xlsx')